# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/30(수) 오전 · requests — 밖에 물어본다</mark>

어제 우리는 수상한 IP 두 개를 찾았습니다. `185.220.101.34` 와 `211.45.12.9` 입니다.

그런데 **그 IP 가 어디 것인지는 우리 파일 어디에도 없습니다.** 오늘은 밖에 물어봅니다.

오늘의 도착점은 **`api_client.py`** 입니다. 조회를 보내고, 실패에 대비하고, 결과를 `api_result.json` 으로 남깁니다.

> 오늘 오후는 **최주용 강사님의 「AI 프리뷰」** 수업입니다. 오전만 이 노트북을 씁니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 내 사본 만들기

위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다. 제목이 「사본: …」으로 바뀌면 된 것입니다.

### 0.2 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 2교시 | 요청을 실제로 보낸다 — `requests.get` · `params` · `headers` |
| 3교시 | 실패에 대비한다 — `timeout` · `raise_for_status` · 재시도 · `.env` |
| 4교시 | `api_client.py` 조립 — `fetch_data()` → `api_result.json` |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 요청을 실제로 보낸다</mark>


아래 두 셀을 먼저 실행합니다. 어제 만든 파일을 여기서 다시 만듭니다. **어제를 못 끝냈어도 여기서부터 시작할 수 있습니다.**


In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 · 오늘 조회할 IP 두 개: 185.220.101.34 · 211.45.12.9")


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **API 조회 서비스** | IP 를 넣으면 무엇을 알려 주는 서비스인가 |
| **`requests`** | 파이썬 기본 도구와 무엇이 다른가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- API 조회 서비스 →
- `requests` →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · 요청을 보내고 응답을 읽는다</mark>


### 왜 필요한가

1. 어제 룰 ②가 `185.220.101.34` 를 잡았습니다. 그런데 **그 IP 가 어느 나라 어느 회사 것인지**는 우리 로그에 없습니다.
2. IP 는 나라와 회사 단위로 배정됩니다. 그래서 **IP 를 넣으면 소유자를 알려 주는 조회 서비스**가 인터넷에 있습니다.
3. 어제는 「서버가 줬다고 치는 값」으로 연습했습니다. 오늘은 **진짜로 보냅니다.**


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| `requests` | 파이썬에서 요청을 보내는 대표 패키지 |
| 외부 라이브러리 | 파이썬에 처음부터 들어 있지 않아 따로 설치하는 도구 |
| `status_code` | 응답에 함께 오는 세 자리 숫자 (어제 배운 상태코드) |
| `.json()` | 응답 본문을 파이썬 값으로 되돌린다 |


### 쓰는 규칙 세 가지

1. **상태코드를 먼저 보고 그다음 본문을 엽니다.** 실패한 응답의 본문은 우리가 바라는 모양이 아닙니다.
2. 주소 뒤에 `?조건=값` 을 손으로 이어 붙이지 않습니다. **`params` 로 넘깁니다.**
3. 인증 정보는 조건이 아니라 **요청의 겉면(`headers`)** 에 담습니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.1 `requests.get` — 한 줄로 보낸다</mark>

```python
import requests

response = requests.get("http://ip-api.com/json/8.8.8.8")

print(response.status_code)     # 200
print(response.json())          # 딕셔너리로 돌아온다
```

- `requests` 는 **외부 라이브러리**입니다. 내 컴퓨터에서는 `pip install requests` 로 깔아야 합니다. **코랩에는 이미 깔려 있습니다.**
- `.json()` 이 돌려주는 것은 **딕셔너리**입니다. 어제 `json.loads` 가 한 일을 대신 해 줍니다.


In [ ]:
import requests  # 코랩에 이미 깔려 있습니다

print("준비 끝")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34")

print(response.status_code)
```

막히면 바로 위 `1.1 requests.get — 한 줄로 보낸다` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34")

print(response.status_code)


✅ `200`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 본문을 열어 한 칸만 꺼냅니다.

```python
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34")
data = response.json()

print(data["country"])
```

막히면 바로 위 `1.1 requests.get — 한 줄로 보낸다` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34")
data = response.json()

print(data["country"])


✅ `Germany`


어제 숫자였던 IP 에 **나라 이름**이 붙었습니다. 우리 파일에 없던 정보가 밖에서 왔습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-3 · 통신사 이름 꺼내기</font></h3></td></tr></table>

같은 주소로 요청을 보내 **통신사 이름**(`isp`)을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Stiftung Erneuerbare Freiheit` |

**💡 힌트**

1. `requests.get(주소)` 로 응답을 받습니다.
2. `.json()` 으로 딕셔너리를 만듭니다.
3. 칸 이름은 `isp` 입니다.


In [ ]:
import requests

url = "http://ip-api.com/json/185.220.101.34"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-4 · 다른 IP 를 조회하기</font></h3></td></tr></table>

어제 룰 ①이 잡은 `211.45.12.9` 를 조회해 **나라와 통신사**를 한 줄로 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `South Korea SamsungSDS Inc` |

**💡 힌트**

1. 주소 끝의 IP 만 바꾸면 됩니다.
2. f-string 으로 IP 를 끼워 넣으면 다음 문제가 쉬워집니다.
3. `print(a, b)` 로 두 값을 한 줄에 냅니다.


In [ ]:
ip = "211.45.12.9"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-5 · 두 IP 를 한 번에</font></h3></td></tr></table>

어제 잡은 IP **두 개**를 차례로 조회해 한 줄씩 출력하시오.

| | |
|---|---|
| 주어지는 값 | `ips = ["185.220.101.34", "211.45.12.9"]` |
| 🎯 나와야 하는 결과 | `[추적] 185.220.101.34 → Germany (Stiftung Erneuerbare Freiheit)` 꼴로 두 줄 |

**💡 힌트**

1. `for` 로 목록을 하나씩 돕니다.
2. 반복 안에서 요청을 보냅니다.
3. f-string 으로 IP·나라·통신사를 한 문장에 넣습니다.


In [ ]:
ips = ["185.220.101.34", "211.45.12.9"]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-1 · 상태코드를 먼저 보기</font></h3></td></tr></table>

응답을 열기 **전에** 상태코드를 확인하고, `200` 일 때만 나라를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Germany` |

**💡 힌트**

1. `response.status_code` 를 `if` 로 견줍니다.
2. 상태코드는 숫자라 따옴표 없이 견줍니다.
3. `200` 이 아니면 코드만 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.2 `params` 와 `headers`</mark>

주소 뒤에 조건을 손으로 이어 붙이면 **빈칸이나 한글이 들어갈 때 주소가 깨집니다.** 딕셔너리로 넘깁니다.

```python
requests.get("http://ip-api.com/json/8.8.8.8", params={"fields": "country,isp"})
```

인증 정보는 조건이 아니라 **겉면**에 담습니다.

```python
requests.get("https://postman-echo.com/headers", headers={"X-Api-Key": "demo-key-1234"})
```

`postman-echo.com/headers` 는 **내가 보낸 겉면을 그대로 비춰 주는 거울** 같은 주소입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-6 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34",
                        params={"fields": "country,isp"})

print(response.json())
```

막히면 바로 위 `1.2 params 와 headers` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34",
                        params={"fields": "country,isp"})

print(response.json())


✅ `{'country': 'Germany', 'isp': 'Stiftung Erneuerbare Freiheit'}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-7 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 겉면에 담아 보낸 것이 그대로 돌아옵니다.

```python
import requests

response = requests.get("https://postman-echo.com/headers",
                        headers={"X-Api-Key": "demo-key-1234"})

print(response.json()["headers"]["x-api-key"])
```

막히면 바로 위 `1.2 params 와 headers` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("https://postman-echo.com/headers",
                        headers={"X-Api-Key": "demo-key-1234"})

print(response.json()["headers"]["x-api-key"])


✅ `demo-key-1234`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-8 · 필요한 칸만 받기</font></h3></td></tr></table>

`211.45.12.9` 를 조회하되 **나라·도시·통신사 세 칸만** 받아 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{'country': 'South Korea', 'city': 'Munjidong', 'isp': 'SamsungSDS Inc'}` |

**💡 힌트**

1. `params` 에 `fields` 라는 칸을 넣습니다.
2. 값은 칸 이름을 쉼표로 이어 붙인 글자입니다.
3. 받은 것을 그대로 출력합니다.


In [ ]:
ip = "211.45.12.9"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-9 · 보낸 겉면 확인하기</font></h3></td></tr></table>

거울 주소로 요청을 보내 **내가 보낸 키가 그대로 돌아오는지** 확인하시오.

| | |
|---|---|
| 주어지는 값 | `token = "sk-0930-demo"` |
| 🎯 나와야 하는 결과 | `sk-0930-demo` |

**💡 힌트**

1. `headers` 에 딕셔너리를 넘깁니다.
2. 칸 이름은 `X-Api-Key` 로 합니다.
3. 돌아온 본문에서 `headers` 안의 `x-api-key` 를 꺼냅니다. 소문자입니다.


In [ ]:
token = "sk-0930-demo"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-10 · 두 IP 를 필요한 칸만</font></h3></td></tr></table>

IP 두 개를 조회하되 **나라와 통신사만** 받아 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `185.220.101.34 Germany Stiftung Erneuerbare Freiheit` 꼴로 두 줄 |

**💡 힌트**

1. 문제 1-5의 반복에 `params` 를 더합니다.
2. `fields` 값은 `country,isp` 입니다.
3. 받은 딕셔너리에서 두 칸을 꺼내 함께 출력합니다.


In [ ]:
ips = ["185.220.101.34", "211.45.12.9"]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-2 · 조회가 실패하는 IP</font></h3></td></tr></table>

`0.0.0.0` 을 조회해 보시오. 이 주소는 실제 장비에 쓰지 않는 주소라 조회가 **실패**합니다. 무엇이 돌아오는지 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `status` 가 `fail` 이고 `message` 에 이유가 담긴 딕셔너리 |

**💡 힌트**

1. 요청 자체는 성공해서 상태코드는 `200` 입니다.
2. **본문 안의 `status` 칸**이 `fail` 입니다.
3. 상태코드만 보면 놓칩니다. 본문도 봐야 합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `requests.get(주소)` | 요청을 보내고 응답을 받는다 |
| `response.status_code` | 어떻게 됐는지 세 자리 숫자 |
| `response.json()` | 본문을 딕셔너리로 되돌린다 |
| `params={...}` | 조건을 안전하게 넘긴다. 주소를 손으로 조립하지 않는다 |
| `headers={...}` | 인증 정보를 요청의 겉면에 담는다 |

⚠ **상태코드가 `200` 이어도 본문이 실패일 수 있습니다.** 도전 1-2 가 그 경우입니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 실패에 대비한다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **`timeout`** | 요청에 시간 제한을 왜 두나 |
| **환경변수** | 비밀 값을 코드 밖에 두면 무엇이 좋아지나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- `timeout` →
- 환경변수 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · 실패는 예외가 아니라 기본값이다</mark>


### 왜 필요한가

1. 네트워크는 자주 끊깁니다. 상대 서버는 점검에 들어가고, 너무 자주 부르면 막습니다.
2. **API 를 부르는 코드에서 성공은 여러 결과 가운데 하나일 뿐입니다.** 실패를 예외 상황으로 보면 코드가 자꾸 멈춥니다.
3. 오늘 교실에서 여러 명이 동시에 조회하면 실제로 막힐 수 있습니다. **그때 쓰는 법을 지금 배웁니다.**


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| `timeout` | 몇 초까지 기다릴지 정하는 값 |
| `raise_for_status` | 상태코드가 4나 5로 시작하면 **예외를 내 준다** |
| `RequestException` | `requests` 가 내는 예외들의 **집안 이름** |
| 환경변수 | 비밀 값을 코드가 아니라 컴퓨터 쪽에 담아 두는 것 |


### 쓰는 규칙 세 가지

1. **`timeout` 을 반드시 붙입니다.** 없으면 상대가 답을 안 줄 때 영원히 기다립니다.
2. `raise_for_status()` 를 불러야 4·5로 시작하는 응답이 **예외가 됩니다.** 그래야 `try` 가 잡습니다.
3. 몇 번 다시 시도해 봅니다. 한 번 실패가 영영 실패는 아닙니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 `timeout` 과 `raise_for_status`</mark>

```python
import requests

response = requests.get("http://ip-api.com/json/8.8.8.8", timeout=3)   # 3초까지만 기다린다
response.raise_for_status()                                            # 4xx·5xx 면 예외
print(response.json()["country"])
```

`raise_for_status()` 는 **성공하면 아무 일도 하지 않습니다.** 실패했을 때만 예외를 냅니다.
예외가 나므로 어제 배운 `try`·`except` 로 감쌉니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import requests

response = requests.get("https://postman-echo.com/status/404", timeout=5)

print(response.status_code)
```

막히면 바로 위 `2.1 timeout 과 raise_for_status` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("https://postman-echo.com/status/404", timeout=5)

print(response.status_code)


✅ `404`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 무엇이 보일까요</font></h3></td></tr></table>

같은 응답에 `raise_for_status()` 를 부르고 `try` 로 감쌌습니다.

```python
import requests

response = requests.get("https://postman-echo.com/status/404", timeout=5)

try:
    response.raise_for_status()
    print("성공")
except requests.RequestException:
    print("요청이 실패했습니다:", response.status_code)
```

막히면 바로 위 `2.1 timeout 과 raise_for_status` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.get("https://postman-echo.com/status/404", timeout=5)

try:
    response.raise_for_status()
    print("성공")
except requests.RequestException:
    print("요청이 실패했습니다:", response.status_code)


✅ `요청이 실패했습니다: 404`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 서버 쪽 오류 잡기</font></h3></td></tr></table>

`https://postman-echo.com/status/500` 에 요청을 보내고, 실패하면 `서버 쪽 문제입니다` 를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `서버 쪽 문제입니다: 500` |

**💡 힌트**

1. `timeout` 을 붙여 요청을 보냅니다.
2. `raise_for_status()` 를 `try` 안에 둡니다.
3. `except requests.RequestException:` 에서 상태코드와 함께 출력합니다.


In [ ]:
import requests

url = "https://postman-echo.com/status/500"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 성공했을 때는 조용하다</font></h3></td></tr></table>

같은 코드를 **성공하는 주소**(`http://ip-api.com/json/185.220.101.34`)로 보내, 성공하면 나라를 출력하시오.

- `raise_for_status()` 는 성공하면 아무 일도 하지 않는다는 것을 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Germany` |

**💡 힌트**

1. 문제 2-3의 코드에서 주소만 바꿉니다.
2. `try` 안에서 `raise_for_status()` 다음에 나라를 출력합니다.
3. `except` 는 그대로 둡니다. 실행되지 않습니다.


In [ ]:
import requests

url = "http://ip-api.com/json/185.220.101.34"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 조회를 함수로 묶기</font></h3></td></tr></table>

IP 하나를 받아 **나라를 돌려주는 함수**를 만드시오. 실패하면 `None` 을 돌려줍니다.

- 함수 이름은 `lookup_country` 로 정합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Germany` · `None` 두 줄 |

**💡 힌트**

1. `def lookup_country(ip):` 로 시작합니다.
2. 성공하면 `return` 으로 나라를 돌려줍니다.
3. `except` 안에서 `return None` 을 합니다. `0.0.0.0` 은 본문이 실패라 `country` 칸이 없습니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-1 · 기다리는 시간을 재 보기</font></h3></td></tr></table>

`timeout` 을 아주 짧게(`0.001` 초) 주면 어떻게 되는지 확인하시오. 답이 오기 전에 포기하게 됩니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `시간 안에 답이 오지 않았습니다` |

**💡 힌트**

1. `timeout=0.001` 처럼 아주 작은 값을 넣습니다.
2. 요청 자체를 `try` 안에 둬야 합니다. 응답을 못 받으니까요.
3. `except requests.RequestException:` 이 시간 초과도 함께 잡습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 `call_with_retry()` 와 `.env`</mark>

한 번 실패가 영영 실패는 아닙니다. **몇 번 다시 보내 봅니다.**

```python
def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            print(f"{i + 1}번째 실패")
    return None
```

- `range(tries)` 는 9/23에 배운 그 `range` 입니다.
- 성공하면 `return` 으로 **바로 빠져나옵니다.** 남은 횟수는 쓰지 않습니다.
- 다 실패하면 `None` 을 돌려줍니다. 부르는 쪽이 `if` 로 확인합니다.

키를 코드에 적으면 깃허브에 올리는 순간 남이 봅니다. **파일에 따로 둡니다.**

```
# .env
API_KEY=demo-key-1234
```

읽는 법은 9/23에 배운 파일 읽기와 `split("=")` 면 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-6 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
for i in range(3):
    print(f"{i + 1}번째 시도")
```

막히면 바로 위 `2.2 call_with_retry() 와 .env` 설명을 다시 봅니다.


In [ ]:
for i in range(3):
    print(f"{i + 1}번째 시도")


✅ `1번째 시도 · 2번째 시도 · 3번째 시도 세 줄`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-7 · 무엇이 보일까요</font></h3></td></tr></table>

글자를 등호로 나눕니다. `.env` 한 줄을 읽는 방법입니다.

```python
line = "API_KEY=demo-key-1234"
parts = line.strip().split("=")

print(parts[1])
```

막히면 바로 위 `2.2 call_with_retry() 와 .env` 설명을 다시 봅니다.


In [ ]:
line = "API_KEY=demo-key-1234"
parts = line.strip().split("=")

print(parts[1])


✅ `demo-key-1234`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-8 · `call_with_retry()` 만들기</font></h3></td></tr></table>

주소를 받아 조회 결과를 돌려주는 **`call_with_retry`** 함수를 만드시오. 실패하면 세 번까지 다시 보냅니다.

- 함수 이름은 학원 교안이 정한 이름이라 그대로 씁니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 성공 주소는 딕셔너리, `status/500` 주소는 세 번 실패 뒤 `None` |

**💡 힌트**

1. `def call_with_retry(url, tries=3):` 로 시작합니다.
2. `for i in range(tries):` 안에 `try` 를 둡니다.
3. 성공하면 `return response.json()`, 다 실패하면 반복 밖에서 `return None` 입니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-9 · `.env` 파일 만들기</font></h3></td></tr></table>

키를 담은 `.env` 파일을 만드시오.

1. 새 코드 셀 맨 첫 줄에 `%%writefile .env` 를 씁니다.
2. 그 아래에 `API_KEY=demo-key-1234` 한 줄을 씁니다.
3. 새 셀에서 `!cat .env` 로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Writing .env` 그리고 `API_KEY=demo-key-1234` |

**💡 힌트**

1. `%%writefile` 은 그 셀을 실행하지 않고 파일로 저장합니다.
2. 키 값은 진짜가 아니어도 됩니다. 오늘은 연습입니다.
3. `!cat` 은 파일 내용을 보여 주는 터미널 명령입니다.


In [ ]:
%%writefile .env


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-10 · `.env` 에서 키 읽기</font></h3></td></tr></table>

방금 만든 `.env` 를 읽어 **키 값만** 꺼내 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `demo-key-1234` |

**💡 힌트**

1. 파일을 한 줄씩 읽는 뼈대는 9/23에 배운 그대로입니다.
2. 줄을 `split("=")` 로 나눕니다.
3. `API_KEY` 로 시작하는 줄만 쓰면 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-11 · `.env.example` 만들기</font></h3></td></tr></table>

`.env.example` 을 만드시오. **값은 비우고 칸 이름만** 남깁니다.

- 진짜 `.env` 는 남에게 주지 않습니다. 대신 **무엇을 채워야 하는지 알려 주는 본보기**를 함께 둡니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `.env.example` 에 `API_KEY=` 한 줄 |

**💡 힌트**

1. 문제 2-9와 같은 방법으로 만듭니다.
2. 파일 이름만 `.env.example` 로 바꿉니다.
3. 등호 뒤를 비워 둡니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-2 · 키를 헤더에 실어 보내기</font></h3></td></tr></table>

`.env` 에서 읽은 키를 **거울 주소**로 보내, 그대로 돌아오는지 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `demo-key-1234` |

**💡 힌트**

1. 문제 2-10으로 키를 먼저 읽습니다.
2. `headers={"X-Api-Key": api_key}` 로 넘깁니다.
3. 돌아온 본문에서 `headers` 안의 `x-api-key` 를 꺼냅니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `timeout=5` | 5초까지만 기다린다. **반드시 붙인다** |
| `raise_for_status()` | 4xx·5xx 면 예외를 낸다. 성공하면 아무 일도 안 한다 |
| `except requests.RequestException` | `requests` 가 내는 예외를 모두 잡는다 |
| `for i in range(tries)` | 몇 번 다시 시도한다 |
| `.env` | 비밀 값을 코드 밖에 둔다 |
| `.env.example` | 값은 비우고 **칸 이름만** 알려 준다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · api_client.py 조립</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **`POST`** | `GET` 과 무엇이 다른가 |
| **`.gitignore`** | 어떤 파일을 왜 적어 두나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- `POST` →
- `.gitignore` →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 조회 결과를 파일로 남긴다</mark>


### 왜 필요한가

1. 조회 결과를 화면에만 찍으면 창을 닫는 순간 사라집니다. 어제 배운 그 이야기입니다.
2. 받은 응답에는 칸이 열네 개나 있습니다. **우리가 쓸 것은 서너 개**입니다. 필요한 것만 골라 남깁니다.
3. 오늘의 산출물은 **`api_client.py`** 와 그것이 남기는 **`api_result.json`** 입니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 `fetch_data()` — 필요한 칸만 고른다</mark>

받은 응답을 그대로 쌓지 않습니다. **쓸 칸만 새 딕셔너리로 옮깁니다.**

```python
def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None
```

이 이름도 학원 교안이 정했습니다. **`call_with_retry`** 와 **`fetch_data`** 두 개입니다.


아래 셀을 먼저 실행합니다. 3교시에서 만든 함수를 여기서 다시 만듭니다.


In [ ]:
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None


print("call_with_retry 준비 끝")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
data = call_with_retry("http://ip-api.com/json/211.45.12.9")

print(len(data))
```

막히면 바로 위 `3.1 fetch_data() — 필요한 칸만 고른다` 설명을 다시 봅니다.


In [ ]:
data = call_with_retry("http://ip-api.com/json/211.45.12.9")

print(len(data))


✅ `14`


칸이 열네 개입니다. 우리가 쓸 것은 셋뿐입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 무엇이 보일까요</font></h3></td></tr></table>

필요한 칸만 새 딕셔너리로 옮깁니다.

```python
data = call_with_retry("http://ip-api.com/json/211.45.12.9")
small = {"ip": data["query"], "country": data["country"], "isp": data["isp"]}

print(small)
```

막히면 바로 위 `3.1 fetch_data() — 필요한 칸만 고른다` 설명을 다시 봅니다.


In [ ]:
data = call_with_retry("http://ip-api.com/json/211.45.12.9")
small = {"ip": data["query"], "country": data["country"], "isp": data["isp"]}

print(small)


✅ `{'ip': '211.45.12.9', 'country': 'South Korea', 'isp': 'SamsungSDS Inc'}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · `fetch_data()` 만들기</font></h3></td></tr></table>

IP 하나를 받아 **세 칸짜리 딕셔너리**를 돌려주는 `fetch_data` 함수를 만드시오. 실패하면 `None` 입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{'ip': '185.220.101.34', 'country': 'Germany', 'isp': 'Stiftung Erneuerbare Freiheit'}` |

**💡 힌트**

1. `call_with_retry` 를 안에서 부릅니다.
2. 돌려받은 것이 있는지 `if data:` 로 확인합니다.
3. 칸 이름은 `ip`·`country`·`isp` 셋입니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 두 IP 를 모아 리스트로</font></h3></td></tr></table>

IP 두 개를 `fetch_data` 로 조회해 **리스트에 모아** 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 딕셔너리 두 개가 담긴 리스트 |

**💡 힌트**

1. 빈 리스트를 반복 전에 만듭니다.
2. `for` 로 IP 를 하나씩 돕니다.
3. `None` 이 아닐 때만 `append` 합니다.


In [ ]:
ips = ["185.220.101.34", "211.45.12.9"]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · 결과를 파일로 남기기</font></h3></td></tr></table>

문제 3-4의 결과를 **`api_result.json`** 으로 저장하시오. 한글 그대로, 두 칸 들여쓰기입니다.

| | |
|---|---|
| 🎯 확인 | `!cat api_result.json` 으로 두 건이 보인다 |

**💡 힌트**

1. 저장은 9/28에 배운 `json.dump` 입니다.
2. `ensure_ascii=False` 와 `indent=2` 를 함께 넣습니다.
3. 파일 이름은 교안이 정한 `api_result.json` 입니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-1 · 추적 문장으로 찍기</font></h3></td></tr></table>

저장한 `api_result.json` 을 읽어 **추적 문장**으로 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[추적] 185.220.101.34 → Germany (Stiftung Erneuerbare Freiheit)` 꼴로 두 줄 |

**💡 힌트**

1. `json.load` 로 읽으면 리스트가 그대로 돌아옵니다.
2. `for` 로 하나씩 돕니다.
3. f-string 으로 세 칸을 한 문장에 넣습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 `api_client.py` 조립 · `POST` 는 이름만</mark>

오늘 만든 조각을 한 파일로 잇습니다. 새 문법은 없습니다.

| 조각 | 어디서 |
|---|---|
| `call_with_retry()` | 3교시 |
| `fetch_data()` | 이 교시 |
| `.env` 읽기 | 3교시 |
| `json.dump` | 9/28 |

**`POST` 는 오늘 손으로 치지 않습니다.** 이름과 차이만 알아 둡니다.

| 메서드 | 무엇 | 보낼 내용을 어디에 |
|---|---|---|
| `GET` | 조회한다 | 주소의 질의(`params`) |
| `POST` | 새로 만든다 | 요청의 **본문**(`json=payload`) |

티켓을 만들어 달라고 할 때처럼 **보낼 내용이 길고 구조가 있을 때** `POST` 를 씁니다. 10/2에 실제로 씁니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-6 · 무엇이 보일까요</font></h3></td></tr></table>

`POST` 로 보낸 내용이 그대로 돌아옵니다. 거울 주소입니다.

```python
import requests

payload = {"title": "의심 IP 확인 요청", "level": "high"}
response = requests.post("https://postman-echo.com/post", json=payload, timeout=5)

print(response.json()["data"])
```

막히면 바로 위 `3.2 api_client.py 조립` 설명을 다시 봅니다.


In [ ]:
import requests

payload = {"title": "의심 IP 확인 요청", "level": "high"}
response = requests.post("https://postman-echo.com/post", json=payload, timeout=5)

print(response.json()["data"])


✅ `{'title': '의심 IP 확인 요청', 'level': 'high'}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-7 · 무엇이 보일까요</font></h3></td></tr></table>

저장한 결과 파일을 열어 봅니다.

```python
!cat api_result.json
```

막히면 바로 위 `3.2 api_client.py 조립` 설명을 다시 봅니다.


In [ ]:
!cat api_result.json


✅ `두 건이 담긴 여러 줄 JSON`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-8 · `api_client.py` 만들기</font></h3></td></tr></table>

오늘의 산출물 **`api_client.py`** 를 만드시오. 새 문법은 없습니다. 조각을 잇는 시간입니다.

1. 새 코드 셀 맨 첫 줄에 `%%writefile api_client.py` 를 씁니다.
2. `import requests` 와 `import json` 을 씁니다.
3. `.env` 에서 `API_KEY` 를 읽습니다.
4. `call_with_retry()` 와 `fetch_data()` 를 그대로 옮깁니다.
5. IP 두 개를 조회해 리스트에 모읍니다.
6. `api_result.json` 으로 저장합니다.
7. 마지막에 추적 문장을 한 줄씩 출력합니다.

| | |
|---|---|
| 🎯 화면 | `[추적] …` 두 줄 |
| 🎯 파일 | `api_result.json` |

**💡 힌트**

1. 어제 만든 `normalize_logs.py` 와 뼈대가 같습니다.
2. `%%writefile` 셀은 실행해도 코드가 돌지 않고 파일로 저장됩니다.
3. 만든 뒤 새 셀에서 `!python api_client.py` 를 실행합니다.


In [ ]:
%%writefile api_client.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-9 · 돌려 보고 확인하기</font></h3></td></tr></table>

만든 파일을 실행하고 결과를 확인하시오.

1. 새 셀에서 `!python api_client.py` 를 실행합니다.
2. 또 새 셀에서 `!cat api_result.json` 으로 내용을 봅니다.

| | |
|---|---|
| 🎯 화면 | `[추적] …` 두 줄과 JSON 두 건 |

**💡 힌트**

1. `!` 는 터미널 명령이라는 표시입니다.
2. 두 명령은 셀을 나눠 실행하는 편이 보기 좋습니다.
3. `.env` 가 같은 폴더에 있어야 실행됩니다.


In [ ]:
!python api_client.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-10 · 드라이브에 남기기</font></h3></td></tr></table>

오늘 산출물 세 개를 내 드라이브 `agent_core` 폴더에 남기시오.

1. 아래 셀로 드라이브를 연결합니다.
2. `agent_core` 폴더로 들어갑니다.
3. 문제 2-9(`.env`)·2-11(`.env.example`)·3-8(`api_client.py`) 셀을 **다시 실행**합니다.
4. `!python api_client.py` 로 확인합니다.

| | |
|---|---|
| 🎯 확인 | `api_client.py` · `.env.example` · `api_result.json` 세 개가 드라이브에 있다 |

**💡 힌트**

1. 폴더를 옮기지 않으면 파일이 코랩 안에만 남습니다.
2. `%cd` 로 폴더를 옮긴 뒤 셀을 다시 실행해야 그 폴더에 저장됩니다.
3. `.env` 는 비밀 파일이라 남에게 주지 않습니다. 드라이브에는 `.env.example` 만 있어도 됩니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-2 · 실패한 IP 도 남기기</font></h3></td></tr></table>

조회에 실패한 IP 도 버리지 않고 `country` 를 `unknown` 으로 채워 남기시오. 어제 배운 정규화와 같은 생각입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 세 건. `0.0.0.0` 은 `country` 가 `unknown` |

**💡 힌트**

1. `fetch_data` 가 `None` 을 돌려줄 때를 `else` 로 받습니다.
2. `else` 에서 `{"ip": ip, "country": "unknown", "isp": "unknown"}` 을 담습니다.
3. 버린 것을 세지 못하면 「왜 두 건이지?」에 답할 수 없습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `fetch_data(ip)` | 조회해서 **쓸 칸만** 새 딕셔너리로 돌려준다 |
| `if data:` | 실패하면 `None` 이라 반드시 확인한다 |
| `json.dump(results, f, …)` | 결과를 파일로 남긴다 |
| `GET` / `POST` | 조회한다 / 새로 만든다 |

오늘 오전의 산출물은 드라이브 `agent_core` 폴더의 **`api_client.py`** · **`.env.example`** · **`api_result.json`** 입니다.

오후는 **최주용 강사님의 「AI 프리뷰」** 수업입니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다.


In [ ]:
#@title 정답 1-3 { display-mode: "form" }
import requests

url = "http://ip-api.com/json/185.220.101.34"

response = requests.get(url)
data = response.json()

print(data["isp"])


In [ ]:
#@title 정답 1-4 { display-mode: "form" }
import requests

ip = "211.45.12.9"

response = requests.get(f"http://ip-api.com/json/{ip}")
data = response.json()

print(data["country"], data["isp"])


In [ ]:
#@title 정답 1-5 { display-mode: "form" }
import requests

ips = ["185.220.101.34", "211.45.12.9"]

for ip in ips:
    response = requests.get(f"http://ip-api.com/json/{ip}")
    data = response.json()
    print(f"[추적] {ip} → {data['country']} ({data['isp']})")


In [ ]:
#@title 정답 ⭐1-1 { display-mode: "form" }
import requests

response = requests.get("http://ip-api.com/json/185.220.101.34")

if response.status_code == 200:
    print(response.json()["country"])
else:
    print("실패:", response.status_code)


In [ ]:
#@title 정답 1-8 { display-mode: "form" }
import requests

ip = "211.45.12.9"

response = requests.get(f"http://ip-api.com/json/{ip}",
                        params={"fields": "country,city,isp"})

print(response.json())


In [ ]:
#@title 정답 1-9 { display-mode: "form" }
import requests

token = "sk-0930-demo"

response = requests.get("https://postman-echo.com/headers",
                        headers={"X-Api-Key": token})

print(response.json()["headers"]["x-api-key"])


In [ ]:
#@title 정답 1-10 { display-mode: "form" }
import requests

ips = ["185.220.101.34", "211.45.12.9"]

for ip in ips:
    response = requests.get(f"http://ip-api.com/json/{ip}",
                            params={"fields": "country,isp"})
    data = response.json()
    print(ip, data["country"], data["isp"])


In [ ]:
#@title 정답 ⭐1-2 { display-mode: "form" }
import requests

response = requests.get("http://ip-api.com/json/0.0.0.0")

print("상태코드:", response.status_code)
print("본문:", response.json())


In [ ]:
#@title 정답 2-3 { display-mode: "form" }
import requests

url = "https://postman-echo.com/status/500"

response = requests.get(url, timeout=5)

try:
    response.raise_for_status()
    print("성공")
except requests.RequestException:
    print("서버 쪽 문제입니다:", response.status_code)


In [ ]:
#@title 정답 2-4 { display-mode: "form" }
import requests

url = "http://ip-api.com/json/185.220.101.34"

response = requests.get(url, timeout=5)

try:
    response.raise_for_status()
    print(response.json()["country"])
except requests.RequestException:
    print("실패:", response.status_code)


In [ ]:
#@title 정답 2-5 { display-mode: "form" }
import requests


def lookup_country(ip):
    response = requests.get(f"http://ip-api.com/json/{ip}", timeout=5)
    try:
        response.raise_for_status()
        data = response.json()
        if data["status"] == "success":
            return data["country"]
        else:
            return None
    except requests.RequestException:
        return None


print(lookup_country("185.220.101.34"))
print(lookup_country("0.0.0.0"))


In [ ]:
#@title 정답 ⭐2-1 { display-mode: "form" }
import requests

try:
    response = requests.get("http://ip-api.com/json/185.220.101.34", timeout=0.001)
    print(response.json()["country"])
except requests.RequestException:
    print("시간 안에 답이 오지 않았습니다")


In [ ]:
#@title 정답 2-8 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            print(f"{i + 1}번째 실패")
    return None


print(call_with_retry("http://ip-api.com/json/185.220.101.34")["country"])
print(call_with_retry("https://postman-echo.com/status/500"))


In [ ]:
#@title 정답 2-9 { display-mode: "form" }
code = "API_KEY=demo-key-1234\n"

with open(".env", "w", encoding="utf-8") as f:
    f.write(code)

print(open(".env", encoding="utf-8").read())


In [ ]:
#@title 정답 2-10 { display-mode: "form" }
api_key = None

with open(".env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=")
        if parts[0] == "API_KEY":
            api_key = parts[1]

print(api_key)


In [ ]:
#@title 정답 2-11 { display-mode: "form" }
with open(".env.example", "w", encoding="utf-8") as f:
    f.write("API_KEY=\n")

print(open(".env.example", encoding="utf-8").read())


In [ ]:
#@title 정답 ⭐2-2 { display-mode: "form" }
import requests

api_key = None
with open(".env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=")
        if parts[0] == "API_KEY":
            api_key = parts[1]

response = requests.get("https://postman-echo.com/headers",
                        headers={"X-Api-Key": api_key}, timeout=5)

print(response.json()["headers"]["x-api-key"])


In [ ]:
#@title 정답 3-3 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None

def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


print(fetch_data("185.220.101.34"))


In [ ]:
#@title 정답 3-4 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None

def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


ips = ["185.220.101.34", "211.45.12.9"]
results = []

for ip in ips:
    row = fetch_data(ip)
    if row:
        results.append(row)

print(results)


In [ ]:
#@title 정답 3-5 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None

import json


def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


results = []
for ip in ["185.220.101.34", "211.45.12.9"]:
    row = fetch_data(ip)
    if row:
        results.append(row)

with open("api_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"{len(results)}건을 저장했습니다")


In [ ]:
#@title 정답 ⭐3-1 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None


def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


import json

results = []
for ip in ["185.220.101.34", "211.45.12.9"]:
    row = fetch_data(ip)
    if row:
        results.append(row)

with open("api_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("api_result.json", encoding="utf-8") as f:
    results = json.load(f)

for row in results:
    print(f"[추적] {row['ip']} → {row['country']} ({row['isp']})")


In [ ]:
#@title 정답 3-8 { display-mode: "form" }
code = """import requests
import json

API_KEY = None
with open(".env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=")
        if parts[0] == "API_KEY":
            API_KEY = parts[1]


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            print(f"{i + 1}번째 실패")
    return None


def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data:
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


results = []
for ip in ["185.220.101.34", "211.45.12.9"]:
    row = fetch_data(ip)
    if row:
        results.append(row)

with open("api_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

for row in results:
    print(f"[추적] {row['ip']} -> {row['country']} ({row['isp']})")
"""

with open("api_client.py", "w", encoding="utf-8") as f:
    f.write(code)

print("api_client.py 를 만들었습니다. 새 셀에서 !python api_client.py 를 실행하세요.")


In [ ]:
#@title 정답 3-9 { display-mode: "form" }
print("아래 두 줄을 각각 새 셀에서 실행합니다.")
print("!python api_client.py")
print("!cat api_result.json")


In [ ]:
#@title 정답 3-10 { display-mode: "form" }
print("!mkdir -p /content/drive/MyDrive/agent_core")
print("%cd /content/drive/MyDrive/agent_core")
print("그다음 .env · .env.example · api_client.py 셀을 다시 실행합니다.")


In [ ]:
#@title 정답 ⭐3-2 { display-mode: "form" }
import requests


def call_with_retry(url, tries=3):
    for i in range(tries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            pass
    return None

def fetch_data(ip):
    data = call_with_retry(f"http://ip-api.com/json/{ip}")
    if data and data["status"] == "success":
        return {"ip": ip, "country": data["country"], "isp": data["isp"]}
    else:
        return None


results = []
for ip in ["185.220.101.34", "211.45.12.9", "0.0.0.0"]:
    row = fetch_data(ip)
    if row:
        results.append(row)
    else:
        results.append({"ip": ip, "country": "unknown", "isp": "unknown"})

for row in results:
    print(row)
